In [0]:
-- create catalog 
CREATE CATALOG IF NOT EXISTS lakehouse_demo;

USE CATALOG lakehouse_demo;

-- create schema
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

-- Simulate a first load from upstream system (e.g a CRM export)
CREATE OR REPLACE TABLE bronze.customers_raw (
  customer_id STRING,
  full_name   STRING,
  email       STRING,
  segment     STRING,  -- e.g. "premium", "standard", "basic"
  country     STRING,
  _source_file STRING, -- which file/batch this came from
  _ingested_at TIMESTAMP -- when we write it into the lakehouse
) USING DELTA;

INSERT INTO bronze.customers_raw VALUES
('C001', 'John Doe', 'john.doe@example.com', 'premium', 'USA', 'customers_20230101.csv', '2024-01-01'),
('C002', 'Ana Guitierrez', 'ana.guitierrez@example.com', 'standard', 'Mexico', 'customers_20230101.csv', '2024-01-01'),
('C003', 'Li Wei', 'li.wei@example.com', 'basic', 'China', 'customers_20230101.csv', '2024-01-01'),
('C003', 'Li Wei', 'li.wei@example.com', 'basic', 'China', 'customers_20230101.csv', '2024-01-01'),
('C004', 'Emily Smith', 'emily.smith@example.com', 'premium', 'UK', 'customers_20230101.csv', '2024-01-01'),
('C005', 'Raj Patel', 'raj.patel@example.com', 'standard', 'India', 'customers_20230101.csv', '2024-01-01'),
('C006', 'Fatima Al-Farsi', 'fatima.alfarsi@example.com', 'basic', 'UAE', 'customers_20230101.csv', '2024-01-01');

-- Read Bronze, cast types, deduplicate, and create the temp view
-- all in one shot using ROW_NUMBER to pick one row per business key

CREATE OR REPLACE TEMP VIEW silver_incoming AS
SELECT
    customer_id,
    full_name,
    email,
    segment,
    country,
    _source_file,
    CAST(_ingested_at AS DATE) AS _ingested_at,
    current_timestamp()        AS _processed_at
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id, full_name, email, segment, country
            ORDER BY _ingested_at DESC   -- if true duplicates exist, keep latest
        ) AS _rn
    FROM bronze.customers_raw
)
WHERE _rn = 1;


CREATE OR REPLACE TABLE lakehouse_demo.silver.customers (
    customer_id STRING,
    full_name STRING,
    email STRING,
    segment STRING,
    country STRING,
    _source_file STRING,
    _ingested_at DATE,
    _processed_at TIMESTAMP,
    valid_from TIMESTAMP,
    valid_to TIMESTAMP,
    is_current BOOLEAN,
    _updated_at TIMESTAMP
) USING DELTA;

-- Insert deduplicated data into silver table
INSERT INTO TABLE lakehouse_demo.silver.customers 
SELECT 
    customer_id,
    full_name,
    email,
    segment,
    country,
    _source_file,
    _ingested_at,
    _processed_at,
    _processed_at AS valid_from,
    NULL AS valid_to,
    true AS is_current,
    current_timestamp() AS _updated_at
FROM silver_incoming;

-- Simulate next Batch landing in Bronze

-- Simulate next batch landing in Bronze
INSERT INTO bronze.customers_raw
  (customer_id, full_name, email, segment, country, _ingested_at)
VALUES
  ('C001', 'Acme Corp', 'acme@example.com', 'enterprise', 'ES', '2024-01-11'),
  ('C003', 'NewCo',     'new@example.com',  'standard',   'IT', '2024-01-11');

  -- SCD Type 2
  --  Step1: Expire ALL current rows for customer_ids in the new batch

  MERGE INTO silver.customers AS target
  USING (
    -- Deduplicate source: keep only latest per customer_id
    SELECT 
      customer_id,
      full_name,
      email,
      segment,
      country,
      _source_file,
      _ingested_at
    FROM (
      SELECT 
        *,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY _ingested_at DESC) AS rn
      FROM bronze.customers_raw
      WHERE `_ingested_at` >= '2024-01-11'
    )
    WHERE rn = 1
  ) AS source
    ON target.customer_id = source.customer_id
    AND target.is_current = true

  -- Unconditionally expire ALL current rows that match (makes pipeline idempotent)
  WHEN MATCHED THEN UPDATE SET
    target.is_current = false,
    target._updated_at = current_timestamp(),
    target.valid_to = source._ingested_at;

    -- Step2 insert new versions

INSERT INTO silver.customers
  (customer_id, full_name, email, segment, country, _source_file, _ingested_at, _processed_at,
   valid_from, valid_to, is_current, _updated_at)
SELECT
    source.customer_id,
    source.full_name,
    source.email,
    source.segment,
    source.country,
    source._source_file,
    CAST(source._ingested_at AS DATE) AS _ingested_at,
    current_timestamp() AS _processed_at,
    source._ingested_at AS valid_from,
    NULL                AS valid_to,
    true                AS is_current,
    current_timestamp() AS _updated_at
FROM (
    -- Deduplicate: same logic as MERGE to ensure consistency
    SELECT 
      customer_id,
      full_name,
      email,
      segment,
      country,
      _source_file,
      _ingested_at
    FROM (
      SELECT 
        *,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY _ingested_at DESC) AS rn
      FROM bronze.customers_raw
      WHERE `_ingested_at` >= '2024-01-11'
    )
    WHERE rn = 1
) AS source
WHERE NOT EXISTS (
    SELECT 1 FROM silver.customers existing
    WHERE existing.customer_id = source.customer_id
      AND existing.valid_from  = source._ingested_at
);

-- Query to check results

-- Current state only
SELECT customer_id, full_name, segment, is_current FROM silver.customers WHERE is_current = true ORDER BY customer_id;

-- Full history for C001
SELECT customer_id, full_name, segment, valid_from, valid_to, is_current
FROM silver.customers
WHERE customer_id = 'C001'
ORDER BY valid_from;

-- Point-in-time: what was C001's segment on 2024-01-10?
SELECT * FROM silver.customers
WHERE customer_id = 'C001'
  AND valid_from <= '2024-01-11'
  AND (valid_to > '2024-01-11' OR valid_to IS NULL)
  ORDER BY valid_from;